# Synthetic Leaf-Only Time Series Data Generation

Generates leaf-level time series with **separate hierarchy columns** — no aggregation.
Use this to test **Skill 1 Case B**: agent detects hierarchy columns and runs `run_aggregation()` in Step 10.

**Output tables:**
- `{catalog}.{schema}.{use_case}_train_data` — leaf series only (`unique_id` = store name, no slashes), with `country`, `region`, `store` columns
- `{catalog}.{schema}.{use_case}_evaluation_output` — synthetic backtest residuals (for MinTrace standalone testing)

## 1. Parameters

In [ ]:
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "catalog_timeseries"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "synthetic"

try:
    use_case = dbutils.widgets.get("use_case")
except Exception:
    use_case = "mmf_leaf"

try:
    n_months = int(dbutils.widgets.get("n_months"))
except Exception:
    n_months = 36

try:
    n_stores = int(dbutils.widgets.get("n_stores"))
except Exception:
    n_stores = 20

train_table = f"{catalog}.{schema}.{use_case}_train_data"
eval_table  = f"{catalog}.{schema}.{use_case}_evaluation_output"
print(f"Train table:  {train_table}")
print(f"Eval table:   {eval_table}")
print(f"Months:       {n_months}")
print(f"Stores:       {n_stores}")

## 2. Generate Leaf-Level Series (no aggregation)

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

REGIONS = [
    ("USA",    "California"),
    ("USA",    "NewYork"),
    ("Europe", "France"),
    ("Europe", "Germany"),
]

leaves = []
for i in range(n_stores):
    country, region = REGIONS[i % len(REGIONS)]
    store = f"Store{i + 1}"
    leaves.append({"country": country, "region": region, "store": store})

bases  = np.random.uniform(100, 300, n_stores)
slopes = np.random.uniform(0.5, 2.0, n_stores)
amps   = np.random.uniform(10,  30,  n_stores)
noises = np.random.uniform(3,   10,  n_stores)

end_date = pd.Timestamp.today().replace(day=1)
dates = pd.date_range(end=end_date, periods=n_months, freq="MS")

rows = []
for i, leaf in enumerate(leaves):
    for t, ds in enumerate(dates):
        y = (
            bases[i]
            + slopes[i] * t
            + amps[i] * np.sin(2 * np.pi * t / 12)
            + np.random.normal(0, noises[i])
        )
        rows.append({
            "unique_id": leaf["store"],
            "ds":        ds.date(),
            "y":         max(0.0, round(y, 4)),
            "country":   leaf["country"],
            "region":    leaf["region"],
            "store":     leaf["store"],
        })

leaf_pdf = pd.DataFrame(rows)
leaf_sdf = spark.createDataFrame(leaf_pdf)

print(f"Leaf series: {leaf_pdf['unique_id'].nunique()} stores x {n_months} months = {len(leaf_pdf):,} rows")
print(f"Date range:  {leaf_pdf['ds'].min()} -> {leaf_pdf['ds'].max()}")
print(f"unique_id sample (no slashes): {sorted(leaf_pdf['unique_id'].unique())[:5]}")
display(leaf_pdf.head(5))

## 3. Save Leaf Data (no aggregation)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

(
    leaf_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(train_table)
)

count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {train_table}").collect()[0]["cnt"]
print(f"Written to {train_table}: {count:,} rows")
print("Columns:", spark.table(train_table).columns)
print("Note: unique_id has NO slashes — Skill 1 will detect 'country','region','store' columns and run aggregation in Step 10")

## 4. Generate Synthetic Evaluation Output (for Skill 6 standalone)

In [ ]:
PREDICTION_LENGTH = 3
MODEL_NAME = "StatsForecastAutoArima"
eval_rows = []

for uid, series in leaf_pdf.groupby("unique_id"):
    series = series.sort_values("ds").reset_index(drop=True)
    n = len(series)
    for offset in [-8, -5]:
        start_idx = n + offset
        end_idx   = start_idx + PREDICTION_LENGTH
        if start_idx < 0 or end_idx > n:
            continue
        window   = series.iloc[start_idx:end_idx]
        actual   = window["y"].tolist()
        forecast = [max(0.0, round(v + np.random.normal(0, abs(v) * 0.05), 4)) for v in actual]
        eval_rows.append({
            "unique_id":                  uid,
            "backtest_window_start_date": pd.to_datetime(window["ds"].iloc[0]),
            "forecast":                   forecast,
            "actual":                     actual,
            "model":                      MODEL_NAME,
        })

eval_pdf = pd.DataFrame(eval_rows)
spark.createDataFrame(eval_pdf).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(eval_table)

print(f"Written to {eval_table}: {len(eval_pdf)} rows across {eval_pdf['unique_id'].nunique()} series")